#  Wprowadzenie

W notebook'ach zdefiniowana jest zmienna `spark` typu `SparkSession`, która będzie punktem startowym naszej pracy ze Sparkiem. Tego typu konwencja nazewnicza jest bardzo częsta.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("pyspark-labs")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 06:56:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Przygotowanie środowiska

Ponizsze fukcji przydadzą nam się do wygodniejszej pracy. Sa powszechne w komercyjnych rozwiązaniach (np. Colab, Databricks).

In [2]:
from IPython.display import display as _display

def display(df):
    _display(df.toPandas())

In [3]:
display(spark.sql("select 1"))

26/04/25 06:56:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


,1
0,1


In [4]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    query = (cell or line or "").strip()
    return display(spark.sql(query))

get_ipython().register_magic_function(sql, "cell")

In [5]:
%%sql
select 1

,1
0,1


# Przygotowanie danych

In [6]:
conf = spark._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(conf)
Path = spark._jvm.org.apache.hadoop.fs.Path

for table in ["uam_offers", "uam_categories", "uam_orders"]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    path = Path(f"s3a://warehouse/{table}")
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(path.toUri(), conf)
    if fs.exists(path):
        fs.delete(path, True)
    spark.read.format("parquet").load(f"s3a://landing-zone/{table}.snappy.parquet").write.mode("overwrite").saveAsTable(table)

26/04/25 06:56:34 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/25 06:56:34 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/04/25 06:56:35 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/04/25 06:56:35 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.19.0.6
26/04/25 06:56:35 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/04/25 06:56:37 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/25 06:56:37 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/04/25 06:56:37 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/25 06:56:37 WARN 

In [7]:
display(spark.sql("select * from uam_offers"))

,offer_id,offer_name,seller_id,starting_time,ending_time,duration,types,buynow_price,auction_minimal_price,auction_starting_price,stock_initial_quantity,stock_current_quantity,category_leaf,attributes,pv
0,004172c9c5049a158006863afb8a3d11,Części samochodowe - BMW E65 3.0D LI,eb4dd0d58f2b23c3963c55f919916fd1,2017-03-06 14:56:46,None,None,[BUY_NOW],170,None,None,1,1,4134,"[(11323, stan, [używany])]",1
1,0081e4d901216584e2fdc42a82ca02b3,Dom i Ogród - Zaślepka zlewu,8573e3a962e9ccd911b4a148cdaeb8f9,2017-08-24 09:24:53,None,None,[BUY_NOW],7,None,None,100000,100000,46133,"[(11323, stan, [nowy]), (17448, waga (z opakow...",1
2,008b51675b5cfc5601ccb2f2d029ef73,Książki i Komiksy - Revised Standar,1c3e9a955223e91cce74b0dc2092a7e8,2017-10-31 07:50:12,None,None,[BUY_NOW],67.07,None,None,15,14,91485,"[(74, rok wydania (xxxx), [2008]), (75, okładk...",1
3,00cb7098fc5f8eea80d32032f60f26f3,Dom i Ogród - MATERAC MADRYT,4daed73eef062490e663639088f5f454,2017-11-18 16:17:11,None,None,[BUY_NOW],799,None,None,1000,1000,20292,"[(11323, stan, [nowy]), (128329, rozmiar (cm),...",0
4,00d100a15d9adaaabdca7ffac0f8d133,Telefony i Akcesoria - PROMOCJA SZKŁO,8af6801901038123f543d393c83f374d,2016-05-18 17:50:49,None,None,[BUY_NOW],3.9,None,None,990,954,10532,"[(11323, stan, [nowy])]",3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,78f88a83250a9bbe9aa086236a5cc870,Kolekcje - Rogatywka WP lą,99b74123d4c0df23c8e27e891318abd4,2017-11-21 22:40:38,None,None,[BUY_NOW],30,None,None,5,3,93556,[],5
184,98ef94e6c5d0e76ae658f1763b796b27,Dom i Ogród - Hygrophila pinn,c7efd0a57cc8b8e85db61df18b71dd4b,2017-12-13 13:58:27,2018-01-12 13:58:27,PT720H,[BUY_NOW],2.95,None,None,6,4,28057,"[(17448, waga (z opakowaniem), [0.5])]",5
185,992f8d5efdddfa828ca5d9713a7f2c2e,Zdrowie - DO47 MILUTKI TE,17c901dee6b0414e722e971d02d5c017,2017-02-08 12:05:28,None,None,[BUY_NOW],6.39,None,None,10000,6160,122587,"[(11323, stan, [nowy])]",31
186,a120d4988021a1bfc465334adfe9d20c,Książki i Komiksy - Gryga - Złoty w,d5cf3b88e9965a41928a9fd45b48d77d,2017-08-25 13:02:10,2018-01-01 00:52:58,None,[BUY_NOW],8,None,None,1,0,79456,"[(11323, stan, [używany])]",1


# Jak tworzymy Dataframe?
Pierwszym sposobem jest użycie Spark SQL:

In [8]:
uam_offers = spark.sql("select * from uam_offers")

W powyższym przypadku prościej będzie jednak skorzystać z metody `table`:

In [9]:
uam_offers = spark.table("uam_offers")

Metoda `sql` przyjmuje dowolne zapytanie Spark SQL. Np.:

In [10]:
spark.sql("show create table uam_offers").collect()

[Row(createtab_stmt='CREATE TABLE spark_catalog.default.uam_offers (\n  offer_id STRING,\n  offer_name STRING,\n  seller_id STRING,\n  starting_time STRING,\n  ending_time STRING,\n  duration STRING,\n  types ARRAY<STRING>,\n  buynow_price STRING,\n  auction_minimal_price STRING,\n  auction_starting_price STRING,\n  stock_initial_quantity INT,\n  stock_current_quantity INT,\n  category_leaf INT,\n  attributes ARRAY<STRUCT<id: STRING, name: STRING, values: ARRAY<STRING>>>,\n  pv BIGINT)\nUSING parquet\n')]

Możemy również odczytać dane bezpośrednio z plików:

In [ ]:
location = "s3a://warehouse/uam_offers"
uam_offers_from_file = spark.read.format("parquet").load(location)

Drugi sposób wymaga od nas znajomości formatu pliku (w typ przypadku `delta`) oraz ścieżki.


Kolejnym sposobem jest wykorzystanie metody `createDataFrame`:

In [ ]:
# Wersja 1:
# Pierwszy argument: lista tupli
# Drugi (opcjonalny) argument: lista string (nazwy kolumn)
spark.createDataFrame([(1, 2), (11, 22)], ['col1', 'col2'])

In [ ]:
# Wersja 2:
# Wykorzystujemy klasę Row, która pozwala nam nazwać odpowiednio kolumny:
from pyspark.sql import Row
spark.createDataFrame([Row(col1 = 1, col2 = 2), Row(col1 = 11, col2 = 22)])

Można korzystać również z źródeł JDBC lub innych dedykowanych bibliotek (MongoDB, Elastic, Cassandra i wiele więcej).


Warto wspomnieć, że metoda `createDataFrame` umożliwia również tworzenie `DataFrame` na bazie `RDD`:

In [ ]:
from pyspark.sql import Row
my_rdd = spark.sparkContext.parallelize([Row(col1 = 1, col2 = 2), Row(col1 = 11, col2 = 22)])
df = spark.createDataFrame(my_rdd)

Można oczywiście dokonać konwersji w drugą stronę:

In [ ]:
df.rdd

Kolejnym sposobem jest przekazanie listy słowników:

In [ ]:
spark.createDataFrame(
  [
    {"col1": 1, "col2": 2},
    {"col1": 11, "col2": 22},
  ]
)

Ćwiczenie


Utwórzmy DataFrame'y *uam_orders* i *uam_categories* analogicznie jak *uam_offers*

Rozwiązanie

In [23]:
uam_orders = spark.table("uam_orders")
uam_categories = spark.table("uam_categories")

Ćwiczenie

Jak utworzyć DF za pomocą metody `createDataFrame` tak, aby zostały nadane domyślne nazwy atrybutów? W jakiej są postaci? Czy nazwy tych atrybutów są czytelne?

Podpowiedź: dokumentację funkcji można sprawdzić dodając `?` na jej końcu (np. `spark.createDataFrame?`).

In [ ]:
spark.createDataFrame?

Rozwiązanie

In [ ]:
spark.createDataFrame([(1, 2), (11, 22)])

#Odczyt metadanych

DF posiada następujące metody i atrybuty umożliwiające eksplorację metadanych:
1. `printSchema()` - wyświetla schemat danych 
2. `schema` - zwraca schemat danych (reprezentacja Spark'owa)
3. `describe(*cols)` - zwraca DF ze statystykami kolumn
4. `dtypes` - zwraca listę par nazwa kolumny, typ kolumny

Klasa `SparkSession` (w naszym przypadku zmienna `spark`) zawiera atrybut `catalog`:

spark.catalog - zwraca obiekt pozwalającym eksplorować metadane schematów i tabel

In [ ]:
uam_offers.printSchema()

In [ ]:
uam_offers.schema

In [ ]:
uam_offers.describe('offer_id', 'seller_id').show(truncate=False)

In [ ]:
uam_offers.dtypes

In [ ]:
spark.catalog.listDatabases()

In [ ]:
spark.catalog.listTables()

In [ ]:
spark.catalog.listColumns('default.uam_categories')

Możemy również wykonać zapytanie SQL:

In [ ]:
%%sql
show tables

# SELECT

Należy zwrócić uwagę, która funkcja zwraca kolumny z przekształcanego DataFrame'a oraz nowe, a która zwraca tylko modyfikowane lub wylistowane kolumny.

1. `uam_categories.select('*')` - wybór wszystkich kolumn (pytanie: czy ma to sens?)
2. `uam_categories.select('category_id', 'category_level1')`  - wybór podzbioru kolumn
3. `uam_categories.select(uam_categories.category_id)` - jak wyżej
4. `uam_categories.select(uam_categories.category_id.alias('id'))` - aliasowanie kolumn
5. `uam_categories.selectExpr('category_id as id', '2*1 as const')` - wyrażenie SQL’owe w klauzuli SELECT
6. `from pyspark.sql.functions import lit; uam_categories.select(lit(2).alias('const'))` - przykład funkcji lit - generowanie stałych
7. `uam_categories.withColumn('const', lit(2))` - dodawanie nowych kolumn
8. `uam_categories.withColumnRenamed('category_id', 'id')` - aliasowanie kolumn - sposób nr 2
9. `uam_categories.drop('category_level3')` - usuwanie kolumn

Niektóre funkcje mogą wymagać przekazania argumentu jako kolumny. Może to być problematyczne np. po operacji złączenia. Rozwiązaniem jest funkcja `col`:

```
from pyspark.sql.functions import col
uam_categories.select(col('category_level1'))
```
Funkcja `col` przyda się w późniejszych przykładach

Zauważmy, że powyższe metody transformują `DataFrame` na inny `DataFrame` bez zwracania wyników (tzw. transformacja). 

Poniższe metody natomiast powodują zwrócenie wyników do `driver`'a (tzw. akcja):
1. `show(n=20, truncate=True, vertical=False)` - wyświetla wyniki na ekran (`n` - liczba rekordów, `truncate` - czy zawijać wiersze, `vertical` - jeśli `True`, to każda kolumna jest wyświetlana w osobny wierszu)
2. `collect()` - zwraca listę `Row`
3. `display(input)` - działa podobnie jak `show`, ale wyniki są wyświetlane w sformatowanej tabeli 

Istnieje oczywiście szereg akcji powodujących zapis danych - umówimy je później

Samodzielne ćwiczenia

Przetestujmy różne polecenia związane z odczytem danych i ich wyświetlaniem poniżej

In [ ]:
# Przykładowo:
uam_categories.select('*').limit(5).show()
uam_categories.select(uam_categories.category_id).limit(5).show()

In [ ]:
uam_categories.select('*').limit(5).collect()

In [ ]:
display(
  uam_categories.select('*').limit(5)
)

## Funkcje wbudowane

UWAGA: Funkcje są podobne jak w przypadku Spark SQL (tj. nazwy, argumenty, działanie)

Docs:
1. https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html

Przykłady znajdują się poniżej:

In [ ]:
# tworzymy DF z jedną kolumną i wierszem
import pyspark.sql.functions as F
from pyspark import Row
df = spark.createDataFrame([Row(col='X')])


In [ ]:
# funkcje operujące na datach
display(
    df.select(
        F.current_timestamp(),
        F.current_date(),
        F.add_months(F.lit("2021-01-01"), 1),
        F.date_add(F.current_date(), 12),
        F.datediff(F.current_date(), F.lit("2021-01-01")),
    )
)

In [ ]:
display(
  df.select(F.lit('1').cast('int'))
)

Należy zwracać uwagę, kiedy funkcja wymaga podania kolumny albo argumentu innego typu. 

Funkcja `expr` jest podobna do funkcji `selectExpr(col*)`. Obydwie umożliwiają do korzystania z funkcji w sposób taki jak korzysta się z nich w Spark SQL. Dobrym przykładem jest `reflect`, którego brakuje w module `pyspark.sql.functions`

In [ ]:
display(df.select(F.expr("reflect('java.util.Currency', 'getAvailableCurrencies')")))

Przejdźmy do bardziej skomplikowanych zastosowań - spróbujemy sparsować dane w formacie JSON, które znajdują się w kolumnie typu String:

Utwórzmy DF z jednym rekordem  i kolumną o nazwie *json*. Korzystając z funkcji *from_json* utworzymy 
z kolumny zawierającej tekst kolumnę sparsowaną (tj. zgodną z podanym schematem):

In [ ]:
import pyspark.sql.types as T
from pyspark import Row
from pyspark.sql.functions import from_json

simple_json = """
{
  "items": [
    {
      "firstItemPrice": {
        "amount": "6.00",
        "currency": "PLN"
      }
    },
    {
      "firstItemPrice": {
        "amount": "8.00",
        "currency": "PLN"
      }
    }
  ]
}
"""
simple_json_df = spark.createDataFrame([Row(json=simple_json)])

Możemy mozolnie utworzyć schemat jak w poniższym przykładzie:

In [ ]:
# tworzymy schemat dla ceny:
price_schema = T.StructType(
    [T.StructField("amount", T.DecimalType(12, 2)), T.StructField("currency", T.StringType())]
)

# schemat dla ceny pojedynczej pozycji
single_item_schema = T.StructType([T.StructField("firstItemPrice", price_schema)])

# schemat dla zbioru różnych pozycji
schema = T.StructType([T.StructField("items", T.ArrayType(single_item_schema))])

# gdy mamy gotowy schemat możemy sparsować nasz rekord:
simple_json_df.select(from_json(simple_json_df.json, schema).alias("parsed_json"))

Ale nie jest to rekomendowane podejście!

Możemy również wygenerować automatycznie schemat dla danych w formacie JSON. W tym celu musimy:

In [ ]:
from pyspark.sql.functions import schema_of_json
json_schema = simple_json_df.select(schema_of_json(simple_json).alias("schema")).head()
json_schema.schema

In [ ]:
simple_json_df.select(from_json(simple_json_df.json, json_schema.schema).alias("parsed_json"))

Wspieranych jest kilka funkcji prefiksowanych `schema_of_...`:

In [ ]:
[entry for entry in F.__dict__.keys() if entry.startswith("schema_of")]

Schemat możemy również wykorzystać tworząc DF przy wykorzystaniu metody `SparkSession.createDataFrame(...)`

In [ ]:
schema = T.StructType(
  [T.StructField("StringField", T.StringType()),
   T.StructField("IntField", T.IntegerType())])
spark.createDataFrame([("val1", 1), ("val2", 2)], schema).printSchema()

Ćwiczenia

Dana jest zmienna tekstowa w formacie JSON (zob. poniżej). Utwórz DF z jednym rekordem i kolumną o nazwie json. Z cennika dostaw pobierz cenę dostawy jednej sztuki z dowolnej metody dostawy.

In [ ]:
json="""
{
  "eventTime": "2018-03-14T03:25:24.516Z",
  "priceList": {
    "shippingRates": null,
    "items": [
      {
        "deliveryMethod": {
          "id": "773167b1-feec-4ae9-b20f-1ed8ccb7b1ed",
          "qeppoId": 6
        },
        "firstItemPrice": {
          "amount": "6.00",
          "currency": "PLN"
        },
        "nextItemPrice": {
          "amount": "0.00",
          "currency": "PLN"
        },
        "packageSize": 5,
        "shippingTime": null
      },
      {
        "deliveryMethod": {
          "id": "758fcd59-fbfa-4453-ae07-4800d72c2ca5",
          "qeppoId": 8
        },
        "firstItemPrice": {
          "amount": "8.00",
          "currency": "PLN"
        },
        "nextItemPrice": {
          "amount": "0.00",
          "currency": "PLN"
        },
        "packageSize": 5,
        "shippingTime": null
      },
      {
        "deliveryMethod": {
          "id": "7203cb90-864c-4cda-bf08-dc883f0c78ad",
          "qeppoId": 9
        },
        "firstItemPrice": {
          "amount": "10.00",
          "currency": "PLN"
        },
        "nextItemPrice": {
          "amount": "0.00",
          "currency": "PLN"
        },
        "packageSize": 10,
        "shippingTime": null
      },
      {
        "deliveryMethod": {
          "id": "845efe05-0c96-47c3-a8cb-aa4699c158ce",
          "qeppoId": 11
        },
        "firstItemPrice": {
          "amount": "12.00",
          "currency": "PLN"
        },
        "nextItemPrice": {
          "amount": "0.00",
          "currency": "PLN"
        },
        "packageSize": 10,
        "shippingTime": null
      }
    ],
    "location": {
      "city": "Internet",
      "state": "zachodniopomorskie",
      "postcode": "00-000",
      "country": "PL"
    },
    "sendingAbroad": false,
    "estimatedShippingTime": "PT0S",
    "additionalInfo": "",
    "id": "5459a18b-4da0-45ef-a910-6472700bde06",
    "lastModified": "2018-03-14T04:25:24.515+01:00",
    "flags": {
      "freeReturn": false,
      "freeDelivery": false,
      "useCostPerWeight": false
    },
    "weight": null,
    "templateId": null,
    "shippingCost": {
      "lowest": {
        "amount": "6.00",
        "currency": "PLN"
      },
      "freeDelivery": false
    }
  },
  "updatedPaths": []
}
"""

In [ ]:
#miejsce na rozwiązanie
#podpowiedź: skorzystać z funkcji  `get_json_object`
spark.sql("desc function get_json_object").show(truncate=False)

Rozwiązanie

In [ ]:
from pyspark import Row
from pyspark.sql.functions import get_json_object

df = spark.createDataFrame([Row(json=json)])
display(
    df.select(get_json_object(df.json, "$.priceList.items[0].firstItemPrice.amount"))
)

Ćwiczenie

Wyświetl liczbę atrybutów dla kilku przykładowych ofert.

In [ ]:
uam_offers.printSchema()

In [ ]:
F.size?

In [ ]:
# miejsce na rozwiązanie
# podpowiedź: skorzystać z funkcji  `size`
display(spark.sql("desc function size"))

Rozwiązanie

In [ ]:
uam_offers.select(uam_offers.offer_id,  F.size(uam_offers.attributes)).show(truncate=False)

Przykład `explode`

Znajdź wszystkie oferty (identyfikator, nazwę oraz typ). Każdy typ powinien się znaleźć w nowym wierszu

In [ ]:
from pyspark.sql.functions import explode

display(
    uam_offers.select(
        explode(uam_offers.types).alias("types_"),
        uam_offers.offer_id,
        uam_offers.offer_name,
    )
)

Rozbicie elementów tablicy na wiersze jest naprostszym sposobem na przetwarzanie elementów. Powoduje to jednak wzrost liczby przetwarzanych wierszy.

Przykład - `transform`

Zamieńmy wielkość liter z wielkich na małe w elementach tablicy `types`:

In [ ]:
display(
  uam_offers.select(
    F.transform("types", F.lower).alias("lowercased_elements"),
    "types"
  ).drop_duplicates()
)


W wyrażeniu lambda możemy podawać wyłącznie funkcje z `pyspark.sql.functions` i UDF'y stworzone w Scali.

Przykład - `filter`

Możemy również odfiltrować elementy tablicy. W poniższym przykładzie wyświetlimy zawartość `attributes` tylko do atrybutów, których nazwa (`name`) równa się "stan": 

In [ ]:
import pyspark.sql.functions as F

display(
  uam_offers.select(
    F.filter("attributes", lambda attrib: attrib["name"] == "stan").alias("stan"),
    "attributes"
  )
)

Ćwiczenie

Za pomocą `filter` wyciągnij wszystkie wartości atrybutów o nazwie `stan` jak w powyższym przykładzie. Następnie za pomocą `transform` wyciagnij wartości (`values`). 

Dla chętnych

Skorzystaj z dodatkowych funkcji `flatten`, aby pozbyć się zagnieżdżonych tablic. Możesz również skorzystać z notacji `[]` do odwołania się do elementu tablicy (podobnie jak w listach)

In [ ]:
import pyspark.sql.functions as F

display(
  uam_offers.select(
      F.flatten(
        F.transform(
          F.filter("attributes", lambda attrib: attrib["name"] == "stan").alias("stan"),
          lambda state: state["values"]
      )
    )[0]
  )
)

## User Defined Function (UDF)

W przypadku, gdy nie ma możliwości skorzystania z funkcji wbudowanych możemy rejestrować własne. Gdy będziemy je wykorzystywać w Spark SQL możemy to zrobić na dwa sposoby:

In [13]:
# Bez określenia zwracanego typu:
def total(price, item_quantity):
    return price * item_quantity


spark.udf.register("total_udf", total)
display(spark.sql("select total_udf(1, 2)"))

,"total_udf(1, 2)"
0,2


In [14]:
# z określeniem zwracanego typu:
from pyspark.sql.types import DecimalType

spark.udf.register("total_udf_typed", total, DecimalType(12, 2))
display(spark.sql("select total_udf_typed(1.0, 2.0)"))

,"total_udf_typed(1.0, 2.0)"
0,2.00


Przykładowe zapytania:

In [15]:
total_query = "select total_udf(unit_price, quantity) from uam_orders"

display(spark.sql(total_query))

,"total_udf(unit_price, quantity)"
0,59.0
1,7.95
2,9.8
3,28.7
4,28.7
...,...
95,36.9
96,21.99
97,34.99
98,17.99


In [16]:
total_typed_query = """
select
  total_udf_typed(
      cast(unit_price as decimal(12, 2)),
      cast(quantity as decimal(12, 2))
    )
from uam_orders"""
display(spark.sql(total_typed_query))

,"total_udf_typed(cast(unit_price as decimal(12,2)), cast(quantity as decimal(12,2)))"
0,59.00
1,7.95
2,9.80
3,28.70
4,28.70
...,...
95,36.90
96,21.99
97,34.99
98,17.99


Ale to nie zadziała zgodnie z oczekiwaniem:

In [17]:
total_typed_query2 = (
    "select total_udf_typed(unit_price,  quantity), price, quantity from uam_orders"
)
display(spark.sql(total_typed_query2))

,"total_udf_typed(unit_price, quantity)",price,quantity
0,None,59.00,1
1,None,7.95,1
2,None,9.80,1
3,None,28.70,1
4,None,28.70,1
...,...,...,...
95,None,36.90,1
96,None,21.99,1
97,None,34.99,1
98,None,17.99,1


Różnicę widać dopiero po wykonaniu `printSchema`

Dla poprawnego wywołania zadziałała niejawna konwersja double->decimal (dot. *quantity*)

W przypadku zapytania, gdzie wynik jest niezgodny z oczekiwanym, mamy mnożenie dwóch liczb typu `double`. Oczekujemy, że funkcja zwróci `decimal`, ale zwraca `double`, co jest konwertowane na wartość `null`.

Jeśli korzystamy z DataFrame API, to mamy następujące możliwości w kwestii UDF:

In [18]:
# 1
from pyspark.sql.functions import udf
def total(price, item_quantity):
    return price * item_quantity

total_typed_udf = udf(total, DecimalType(12,2))

In [21]:
#2 dekorator udf
from pyspark.sql.functions import udf

@udf('decimal(12,2)')
def total_typed_udf(price, item_quantity):
    return price * item_quantity

W obydwu przypadkach możemy pominąć typ zwracany przez UDF:

`total_udf = udf(total)`

Lub w wersji z dekoratorem:

```
@udf
def total_udf(price, item_quantity):
    return price * item_quantity * 1.0
```

Ostatecznie możemy skorzystać analogicznie jak z innych funkcji:

In [24]:
display(
    uam_orders.select(
        total_typed_udf(
            uam_orders.price.cast("decimal(12,2)"),
            uam_orders.quantity.cast("decimal(12,2)"),
        )
    )
)

,"total_typed_udf(cast(price as decimal(12,2)), cast(quantity as decimal(12,2)))"
0,59.00
1,7.95
2,9.80
3,28.70
4,28.70
...,...
95,36.90
96,21.99
97,34.99
98,17.99


UDF - przykład

Napisz funkcję, która dla 3 poziomów drzewa kategorii utworzy jedną nazwę. 

W przypadku, gdy kategoria na danym poziomie nie jest wypełniona należy użyć nazwy kategorii na wyższym poziomie. Jeśli kategoria w ogóle nie jest wypełniona, to można wpisać dowolny ciąg znaków. Przykłady:

*Elektronika - RTV i AGD - RTV i AGD*

*Elektronika - Elektronika - Elektronika*

*Brak nazwy - Brak nazwy - Brak nazwy*

*Kolekcje i sztuka - Kolekcje - Pocztówki *

In [25]:
@udf
def format_category_tree(cat1, cat2, cat3):
    cat1_or_default = cat1 or "Brak nazwy"
    cat2_or_default = cat2 or cat1_or_default
    cat3_or_default = cat3 or cat2_or_default
    return cat1_or_default + " - " + cat2_or_default + " - " + cat3_or_default


display(
    uam_categories.select(
        format_category_tree(
            uam_categories.category_level1,
            uam_categories.category_level2,
            uam_categories.category_level3,
        )
    )
)

,"format_category_tree(category_level1, category_level2, category_level3)"
0,Kultura i rozrywka - Muzyka - Płyty winylowe
1,Kultura i rozrywka - Książki i Komiksy - Liter...
2,Moda i uroda - Uroda - Pielęgnacja
3,Dom i zdrowie - Dom i Ogród - Narzędzia
4,Kultura i rozrywka - Książki i Komiksy - Czaso...
...,...
169,Dom i zdrowie - Delikatesy - Dekoracje cukiern...
170,Elektronika - Telefony i Akcesoria - Akcesoria...
171,Firma i usługi - Przemysł - Odzież robocza i BHP
172,Sport i wypoczynek - Sport i Turystyka - Bieganie


Zadanie

Zamień ciągi znaków w nazwach ofert tak aby były z wielkich liter:
1) Skorzystaj z funkcji wbudowanych
2) Stwórz UDF'a

In [26]:
from pyspark.sql.functions import upper

display(uam_offers.select(upper(uam_offers.offer_name)))

,upper(offer_name)
0,CZĘŚCI SAMOCHODOWE - BMW E65 3.0D LI
1,DOM I OGRÓD - ZAŚLEPKA ZLEWU
2,KSIĄŻKI I KOMIKSY - REVISED STANDAR
3,DOM I OGRÓD - MATERAC MADRYT
4,TELEFONY I AKCESORIA - PROMOCJA SZKŁO
...,...
183,KOLEKCJE - ROGATYWKA WP LĄ
184,DOM I OGRÓD - HYGROPHILA PINN
185,ZDROWIE - DO47 MILUTKI TE
186,KSIĄŻKI I KOMIKSY - GRYGA - ZŁOTY W


In [27]:
from pyspark.sql.functions import udf


@udf
def to_upper(s: str):
    return s.upper()


display(uam_offers.select(to_upper(uam_offers.offer_name)))

,to_upper(offer_name)
0,CZĘŚCI SAMOCHODOWE - BMW E65 3.0D LI
1,DOM I OGRÓD - ZAŚLEPKA ZLEWU
2,KSIĄŻKI I KOMIKSY - REVISED STANDAR
3,DOM I OGRÓD - MATERAC MADRYT
4,TELEFONY I AKCESORIA - PROMOCJA SZKŁO
...,...
183,KOLEKCJE - ROGATYWKA WP LĄ
184,DOM I OGRÓD - HYGROPHILA PINN
185,ZDROWIE - DO47 MILUTKI TE
186,KSIĄŻKI I KOMIKSY - GRYGA - ZŁOTY W


UDF a wydajność:

UDF w PySpark są wolniejsze niż w Scali oraz Javie ze względu na konieczność serializacji i deserializacji danych przy wykonywaniu funkcji pythonowych (dwukrotnie). 
https://blog.cloudera.com/blog/2017/02/working-with-udfs-in-apache-spark/

Od wersji 2.3 wprowadzono tzw. Pandas UDF, zaś począwszy od wersji 3.0 wprowadzone nowe uporządkowane API:

https://databricks.com/blog/2020/05/20/new-pandas-udfs-and-python-type-hints-in-the-upcoming-release-of-apache-spark-3-0.html

Przykład znajduje się poniżej:

In [ ]:
from pyspark.sql.functions import pandas_udf, PandasUDFType
import pandas as pd


@pandas_udf("decimal(12,2)")
def total_typed_udf(price: pd.Series, item_quantity: pd.Series) -> pd.Series:
    return price * item_quantity


display(
    uam_orders.select(
        total_typed_udf(
            uam_orders.price.cast("decimal(12,2)"),
            uam_orders.quantity.cast("decimal(12,2)"),
        )
    )
)

Od najnowszej wersji Spark'a (3.5) dostępne jest kolejne usprawnienie, które rozwiązuje problem z podwójną serializacją/deserializacją - *Arrow Optimized UDFs:* https://www.databricks.com/blog/arrow-optimized-python-udfs-apache-sparktm-35

W tym przypadku wystarczy tylko:

In [ ]:
@udf(useArrow=True)
def total_arrow_udf(price, item_quantity):
    return price * item_quantity

In [ ]:
display(
    uam_orders.select(
        total_arrow_udf(
            uam_orders.price.cast("decimal(12,2)"),
            uam_orders.quantity.cast("decimal(12,2)"),
        )
    )
)

Możemy sprawdzić plan zapytania:

In [ ]:
uam_orders.select(
        total_arrow_udf(
            uam_orders.price.cast("decimal(12,2)"),
            uam_orders.quantity.cast("decimal(12,2)"),
        )
    ).explain()

# WHERE

Działanie metody `where` jest bardzo intuicyjne:

In [ ]:
# Znajdźmy transakcje dokonane po 2018-01-01 23:00:00
display(uam_orders.where(uam_orders.buyingTime > "2018-01-01 23:00:00"))

In [ ]:
# Znajdźmy oferty z jedną odsłoną
display(uam_offers.where(uam_offers.pv == 1))

Intuicyjne również są operatory logiczne, chociaż warto zwrócić uwagę, na potrzebę umieszczania wyrażeń w nawiasach ze względu na kolejność wykonywania operatorów:

In [ ]:
# Koniunkcja

# Znajdźmy kategorie, które na 3. poziomie mają wartość różną od NULL, a na 1. "Moda i uroda"
display(
    uam_categories.where(
        (uam_categories.category_level3.isNotNull())
        & (uam_categories.category_level1 == "Moda i uroda")
    )
)

In [ ]:
# Alternatywa

# Znajdźmy oferty z jedną odsłoną lub liczbą odsłon większą niż 10
display(uam_offers.where((uam_offers.pv == 1) | (uam_offers.pv > 10)))

Istnieje wiele funkcji dostępnych na kolumnach, które zwracają typ `boolean`. Np.:

In [ ]:
# Znajdźmy kategorie na 1. poziomie, które nie składają się ze słów rozdzielonych spójnikiem “i”
display(uam_categories.where(~uam_categories.category_level1.like("% i %")))

Kolejnym przykładem takich funkcji są `isNull()` oraz `isNotNull()`:

In [ ]:
display(uam_orders.where(uam_orders.userAgent.isNull()))
display(uam_orders.where(uam_orders.userAgent.isNotNull()))

Intuicyjne są również operatory porównania:

In [ ]:
uam_offers.where(uam_offers.pv > 1)
uam_offers.where(uam_offers.pv >= 1)
uam_offers.where(uam_offers.pv < 1)
uam_offers.where(uam_offers.pv <= 1)
uam_offers.where(uam_offers.pv != 1)

Przykład

Znajdźmy identyfikatory ofert wraz z dostępnymi rozmiarami. Informacje  o rozmiarach znajdują się w kolumnie *attributes*. Rozmiar jest jednym z rodzajów atrybutów.

In [ ]:
from pyspark.sql.functions import explode, col

display(
    uam_offers.select(
        uam_offers.offer_id,
        explode(uam_offers.attributes).alias("attribute")
    )
    .where(col("attribute.name") == "rozmiar")
    .select(
        uam_offers.offer_id,
        explode(col("attribute.values")).alias("attribute_value")
    )
)

Zadanie 

Znajdźmy oferty "Kup teraz" (typ zawiera BUY_NOW), które są niedostępne (tj. stan magazynowy *stock_current_quantity* jest równy 0).

In [28]:
uam_offers.printSchema()

root
 |-- offer_id: string (nullable = true)
 |-- offer_name: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- starting_time: string (nullable = true)
 |-- ending_time: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- types: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- buynow_price: string (nullable = true)
 |-- auction_minimal_price: string (nullable = true)
 |-- auction_starting_price: string (nullable = true)
 |-- stock_initial_quantity: integer (nullable = true)
 |-- stock_current_quantity: integer (nullable = true)
 |-- category_leaf: integer (nullable = true)
 |-- attributes: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- values: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |-- pv: long (nullable = true)



In [38]:
# display(uam_offers.select("*").limit(1))
from pyspark.sql.functions import array_contains
display(
    uam_offers.where(
        (array_contains(uam_offers.types, "BUY_NOW"))
        & (uam_offers.stock_current_quantity == 0)
    )
)

,offer_id,offer_name,seller_id,starting_time,ending_time,duration,types,buynow_price,auction_minimal_price,auction_starting_price,stock_initial_quantity,stock_current_quantity,category_leaf,attributes,pv
0,01b9785065b02fd118062fc0d9bde9ed,Książki i Komiksy - INFERNO - DAN B,4d06c4f2d4972fb6ac47d23a9fb0c6a4,2017-12-18 09:00:00,2018-01-01 23:29:48,None,[BUY_NOW],7.99,None,None,1,0,92475,"[(11323, stan, [używany]), (17448, waga (z opa...",2
1,222200c5287e379585e20ee36d04635f,Kolekcje - 2010 NOW! IGNAC,d3e462b24ac7342c45f65f6cd989a20a,2017-12-31 21:38:22,2018-01-01 16:41:49,PT720H,[BUY_NOW],4.75,None,None,36,0,17886,"[(213, rodzaj, [czysty])]",2
2,549f336548f8937231a1ed543dfa7daf,Komputery - HP 4540s klapa,9e947cc059cba3257071c3e2430e5ed0,2017-12-23 19:50:52,2018-01-01 23:38:57,None,[BUY_NOW],25,None,None,1,0,77856,"[(11323, stan, [używany])]",1
3,e37e354798ea019b30fb306b2eea29d2,"Odzież, Obuwie, Dodatki - *72556CS TRAFAL",3ed2543f75da357c7c9c882aebe8b46c,2017-12-31 21:16:11,2018-01-01 10:25:12,PT168H,"[AUCTION, BUY_NOW]",44.99,0,31.49,1,0,76034,"[(59, dekolt, [okrągły]), (451, kolor, [odcien...",0
4,e4d9e1966846001657875cb269684262,Odzież - NEXT * Bawełnia,45062781b90e97f3e379e41a17de57ae,2017-11-24 14:45:37,2018-01-01 18:41:15,None,[BUY_NOW],19.99,None,None,1,0,89316,"[(11323, stan, [nowy]), (7174, marka, [next]),...",0
5,ee284ff6c7e02fc2af4183b93002564f,"Odzież, Obuwie, Dodatki - NNJ358* H&M SPO",d1d0b438630fab71dea18664a6f545e5,2017-12-29 14:58:25,2018-01-01 14:29:32,PT72H,[BUY_NOW],9.99,None,None,1,0,87842,"[(11323, stan, [używany]), (15851, płeć, [prod...",0
6,ffa338100e36e0800fbafdb02b61a8ec,Uroda - Tarte Shape Tap,2d280e26fc45934a4148444e91e251fb,2017-12-28 21:04:51,2018-01-01 15:57:36,None,[BUY_NOW],139,None,None,3,0,89011,"[(11323, stan, [nowy]), (129369, marka, [inna ...",5
7,2c03ad175869262ac79ab1604202f8a5,Zabawki - ALVI MATA DO RA,a8477f89e3b1547f589894c13274ba35,2017-12-21 12:13:33,2018-01-01 16:57:48,PT480H,[BUY_NOW],65,None,None,1,0,93578,"[(11323, stan, [nowy]), (127812, marka, [inna]...",1
8,f63474426d313d4cb1877318da203ad7,Książki i Komiksy - Elektronika dla,6bfa161f432c7ba544647e7c2603b690,2017-12-29 13:06:26,2018-01-01 16:52:46,None,[BUY_NOW],1,None,None,1,0,754,"[(11323, stan, [używany]), (17448, waga (z opa...",1
9,ff384c714e246f1f9833bd4a93feb1c1,"Odzież, Obuwie, Dodatki - 21F089 NN6 PLIS",d52348ca463cf4077b97514532bbb99b,2017-12-31 21:41:37,2018-01-01 15:35:11,PT168H,"[AUCTION, BUY_NOW]",68.99,0,28.99,1,0,124264,"[(451, kolor, [wielokolorowy]), (3766, wzór do...",2


Rozwiązania

In [ ]:
from pyspark.sql.functions import array_contains

display(
    uam_offers.where(
        (array_contains(uam_offers.types, "BUY_NOW"))
        & (uam_offers.stock_current_quantity == 0)
    ).select("offer_id")
)

In [39]:
from pyspark.sql.functions import col, explode

display(
    uam_offers.withColumn("type", explode(uam_offers.types))
    .where((col("type") == "BUY_NOW") & (uam_offers.stock_current_quantity == 0))
    .select("offer_id")
)

,offer_id,offer_name,seller_id,starting_time,ending_time,duration,types,buynow_price,auction_minimal_price,auction_starting_price,stock_initial_quantity,stock_current_quantity,category_leaf,attributes,pv,type
0,004172c9c5049a158006863afb8a3d11,Części samochodowe - BMW E65 3.0D LI,eb4dd0d58f2b23c3963c55f919916fd1,2017-03-06 14:56:46,None,None,[BUY_NOW],170,None,None,1,1,4134,"[(11323, stan, [używany])]",1,BUY_NOW
1,0081e4d901216584e2fdc42a82ca02b3,Dom i Ogród - Zaślepka zlewu,8573e3a962e9ccd911b4a148cdaeb8f9,2017-08-24 09:24:53,None,None,[BUY_NOW],7,None,None,100000,100000,46133,"[(11323, stan, [nowy]), (17448, waga (z opakow...",1,BUY_NOW
2,008b51675b5cfc5601ccb2f2d029ef73,Książki i Komiksy - Revised Standar,1c3e9a955223e91cce74b0dc2092a7e8,2017-10-31 07:50:12,None,None,[BUY_NOW],67.07,None,None,15,14,91485,"[(74, rok wydania (xxxx), [2008]), (75, okładk...",1,BUY_NOW
3,00cb7098fc5f8eea80d32032f60f26f3,Dom i Ogród - MATERAC MADRYT,4daed73eef062490e663639088f5f454,2017-11-18 16:17:11,None,None,[BUY_NOW],799,None,None,1000,1000,20292,"[(11323, stan, [nowy]), (128329, rozmiar (cm),...",0,BUY_NOW
4,00d100a15d9adaaabdca7ffac0f8d133,Telefony i Akcesoria - PROMOCJA SZKŁO,8af6801901038123f543d393c83f374d,2016-05-18 17:50:49,None,None,[BUY_NOW],3.9,None,None,990,954,10532,"[(11323, stan, [nowy])]",3,BUY_NOW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,98ef94e6c5d0e76ae658f1763b796b27,Dom i Ogród - Hygrophila pinn,c7efd0a57cc8b8e85db61df18b71dd4b,2017-12-13 13:58:27,2018-01-12 13:58:27,PT720H,[BUY_NOW],2.95,None,None,6,4,28057,"[(17448, waga (z opakowaniem), [0.5])]",5,BUY_NOW
189,992f8d5efdddfa828ca5d9713a7f2c2e,Zdrowie - DO47 MILUTKI TE,17c901dee6b0414e722e971d02d5c017,2017-02-08 12:05:28,None,None,[BUY_NOW],6.39,None,None,10000,6160,122587,"[(11323, stan, [nowy])]",31,BUY_NOW
190,a120d4988021a1bfc465334adfe9d20c,Książki i Komiksy - Gryga - Złoty w,d5cf3b88e9965a41928a9fd45b48d77d,2017-08-25 13:02:10,2018-01-01 00:52:58,None,[BUY_NOW],8,None,None,1,0,79456,"[(11323, stan, [używany])]",1,BUY_NOW
191,e19407963c08126e9f36bd669e0431e5,"Odzież, Obuwie, Dodatki - 26Q042 MAYA TAL",d52348ca463cf4077b97514532bbb99b,2017-12-25 22:17:21,2018-01-01 22:17:21,PT168H,"[AUCTION, BUY_NOW]",144.99,0,54.99,1,0,124264,"[(451, kolor, [odcienie zieleni]), (3766, wzór...",11,AUCTION


Aliasem dla metody `where` jest `filter`:

In [ ]:
display(
    uam_offers.withColumn("type", explode(uam_offers.types))
    .filter((col("type") == "BUY_NOW") & (uam_offers.stock_current_quantity == 0))
    .select("offer_id")
)

Do metod `where`/`filter` możemy przekazywać warunki w postaci klauzul SQL, tj.:

In [ ]:
display(
    uam_offers.withColumn("type", explode(uam_offers.types))
    .where("type = 'BUY_NOW' and stock_current_quantity = 0")
    .select("offer_id")
)

Wracamy do funkcji wyższego rzędu - uzupełnienie `transform` i `filter`


Przykład - `exists`

Możemy również odfiltrować elementy tablicy. W poniższym przykładzie ograniczymy zawartość `attributes` tylko do atrybutów, których nazwa (name) równa się "stan":

In [ ]:
from pyspark.sql.functions import exists

display(
  uam_offers.where(
    exists("attributes", lambda attrib: attrib["name"] == "stan").alias("stan"),
  )
)

# JOIN

Do operacji złączenia służy metoda `join` zdefiniowana na DataFrame:

`join(other, on=None, how=None)`

Znaczenie parametrów:
1. `other` - DF, z którym przeprowadzamy operację złączenia (prawa strona)
2. `on` - string, lista kolumn lub wyrażenie bazujące na kolumnach
3. `how` -  *inner* (domyślnie), *cross, outer, full, full_outer, left, left_outer, right, right_outer, left_semi, and left_anti*

Przykład 

Połącz oferty z kategoriami. Wybierz tylko te oferty, które są droższe niż 100 zł i należą do kategorii "Kolekcje i sztuka".

In [40]:
display(
    uam_offers.join(
        uam_categories, uam_offers.category_leaf == uam_categories.category_id
    )
    .where(
        (uam_offers.buynow_price > 100)
        & (uam_categories.category_level1 == "Kolekcje i sztuka")
    )
    .select(
        uam_offers.offer_id,
        uam_offers.offer_name,
        uam_categories.category_level1,
        uam_categories.category_level2,
        uam_categories.category_level3,
    )
)

,offer_id,offer_name,category_level1,category_level2,category_level3
0,01dcb1b3bf7113eadb4c7d6961c4aa10,Kolekcje - Model do skleja,Kolekcje i sztuka,Kolekcje,Modelarstwo


Ćwiczenie

Sporządź zestawienie ofert sprzedanych danego dnia (identyfikator, nazwa). Wybierz tylko te oferty, które są droższe niż 1000 zł.

In [43]:
display(uam_offers.where(uam_offers.ending_time == '2018-01-12').select('*'))

,offer_id,offer_name,seller_id,starting_time,ending_time,duration,types,buynow_price,auction_minimal_price,auction_starting_price,stock_initial_quantity,stock_current_quantity,category_leaf,attributes,pv


In [50]:
display(
    uam_offers.join(uam_orders, uam_orders.offer_id == uam_offers.offer_id)
    .where(uam_offers.buynow_price > 1000)
    .select(uam_offers.offer_id, uam_offers.offer_name)
)

,offer_id,offer_name
0,37ff8bac91526673a489c4defff9d55d,Telefony i Akcesoria - Iphone 6S 16GB


Rozwiązania

In [51]:
display(
    uam_offers.join(uam_orders, uam_offers.offer_id == uam_orders.offer_id)
    .where(uam_offers.buynow_price > 1000)
    .select(uam_offers.offer_id, uam_offers.offer_name)
)

display(
    uam_offers.join(uam_orders, "offer_id")
    .where(uam_offers.buynow_price > 1000)
    .select(uam_offers.offer_id, uam_offers.offer_name)
)

display(
    uam_offers.join(uam_orders, ["offer_id"])
    .where(uam_offers.buynow_price > 1000)
    .select(uam_offers.offer_id, uam_offers.offer_name)
)

,offer_id,offer_name
0,37ff8bac91526673a489c4defff9d55d,Telefony i Akcesoria - Iphone 6S 16GB


,offer_id,offer_name
0,37ff8bac91526673a489c4defff9d55d,Telefony i Akcesoria - Iphone 6S 16GB


,offer_id,offer_name
0,37ff8bac91526673a489c4defff9d55d,Telefony i Akcesoria - Iphone 6S 16GB


# Grupowanie i agregacja

Odpowiednikiem klauzuli `GROUP BY` jest DataFrame'owa metoda `groupBy(*cols)`, natomiast do wyliczania agregatów służy metoda `agg(*exprs)` zdefiniowana w klasie `GroupedData`. 

Dla hipotycznego DataFrame'a *df* kombinacja wywołań w/w metod będzie następująca:

`df.groupBy(*cols).agg(*exprs)`, gdzie

1. `*cols` - lista kolumn lub string (nazwy kolumn)
2. `*exprs` - lista wyrażeń grupujących kolumny lub słownik, gdzie kluczem jest nazwa kolumny a wartością funkcja grupująca

Przykład

Znajdź maksymalną i minimalną cenę ofert w każdej z kategorii na 1. poziomie

In [ ]:
from pyspark.sql.functions import min, max

display(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .groupBy(uam_categories.category_level1)
    .agg(
        min(uam_offers.buynow_price.cast("double")).alias("min_price"),
        max(uam_offers.buynow_price.cast("double")).alias("max_price"),
    )
)

In [ ]:
# inne rozwiązania:

# lista kolumn w groupBy
display(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .groupBy([uam_categories.category_level1])
    .agg(
        min(uam_offers.buynow_price.cast("double")),
        max(uam_offers.buynow_price.cast("double")),
    )
)

# string w groupBy
display(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .groupBy("uam_categories.category_level1")
    .agg(
        min(uam_offers.buynow_price.cast("double")),
        max(uam_offers.buynow_price.cast("double")),
    )
)

# lista string'ów w groupBy
display(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .groupBy(["uam_categories.category_level1"])
    .agg(
        min(uam_offers.buynow_price.cast("double")),
        max(uam_offers.buynow_price.cast("double")),
    )
)

# słownik w agg
display(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .withColumn("buynow_price_as_double1", uam_offers.buynow_price.cast("double"))
    .withColumn("buynow_price_as_double2", uam_offers.buynow_price.cast("double"))
    .groupBy("uam_categories.category_level1")
    .agg({"buynow_price_as_double1": "min", "buynow_price_as_double2": "max"})
)

Funkcje grupujące nazywają się podobnie jak w Spark SQL (różniące się zostały pogrubione):

*avg*

*collect_list*

*collect_set*

*corr*

*first*

*kurtosis*

*last*

*max*

*mean*

*min*

*skewness*

*stddev_pop*

*stddev_samp*

*stddev*

*var_pop*

*var_samp*

*variance*

**approx_count_distinct**

*count*

**countDistinct**

*sum*

**sumDistinct**

Niektóre funkcje grupujące są również metodami `pyspark.sql.group.GroupedData` (tj. pomijamy wywołanie `agg(...)`), np.:

In [ ]:
gdf = uam_offers.groupBy(uam_offers.duration)
gdf.max("pv", "stock_initial_quantity")
# Pozostałe funkcje:
gdf.avg("pv", "stock_initial_quantity")
gdf.count()
gdf.mean("pv", "stock_initial_quantity")
gdf.min("pv", "stock_initial_quantity")
gdf.sum("pv", "stock_initial_quantity")

Funkcja `pivot` umożliwia zamianę wartości w wierszach wybranej kolumny na nazwę nowych kolumn oraz przeprowadzenie odpowiedniej agregacji. Przykładowo, jeśli chcemy pogrupować sumę odsłon oferty w przekroju kategorii (1. poziom) oraz czasu trwania, tak aby czas trwania był nowych kolumnach, to musimy:

In [ ]:
df = uam_offers.join(
    uam_categories, uam_categories.category_id == uam_offers.category_leaf
)
display(
    df.groupBy(df.category_level1)
    .pivot("duration", values=["PT168H", "PT240H"])
    .sum("pv")
)

Uwaga: funkcja pivot przyjmuje opcjonalny argument `values`. Jeśli jest wypełniony, to zostaną utworzone te kolumny, które znajdują się na liście. 
Pytanie: która wersja pivot jest wydajniejsza? Dlaczego?

In [ ]:
display(df.groupBy(df.category_level1).pivot("duration").sum("pv"))

Grupowanie (`cube/rollup`) oraz sortowanie - przykład

Znajdźmy sumę odsłon oraz liczbę ofert w podziale na kategorię (1. poziom) oraz czas trwania oferty. Posortujmy wyniki ze względu na poziom grupowania:

In [ ]:
from pyspark.sql.functions import grouping_id, grouping, sum, count, col

df = uam_offers.join(
    uam_categories, uam_categories.category_id == uam_offers.category_leaf
)

display(
    df.cube("category_level1", "duration")
    .agg(
        grouping_id(),
        grouping("category_level1"),
        grouping("duration"),
        sum(df.pv),
        count("*"),
    )
    .orderBy(col("grouping_id()"))
)

W odróżnieniu od Spark SQL nie ma odpowiednika funkcji `grouping sets` (tzn. jest dostępna implementacja w Sparku 4.0 w wersji [preview](https://spark.apache.org/docs/4.0.0-preview2/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupingSets.html)).

Dodatkowe funkcje `grouping_id()` oraz `grouping(col)` służą odpowiednio do określania poziomu agregacji oraz określenia, czy wartość w kolumnie jest wynikiem agregacji, czy jest wartością występującą w zbiorze wynikowym (jest to szczególnie ważne w przypadku wartości *NULL*)

Funkcja `rollup` w odróżnieniu od `cube` nie generuje wszystkich możliwych podsumowań, ale takie które wynikają z kolejności podanych kolumn. Np. `rollup(col1, col2)` zwróci całościowe podsumowanie oraz na poziomie kolumny *col1* oraz pary *(col1, col2)*. Nie zostanie wygenerowane podsumowanie na poziomie *col2*.

# Funkcje analityczne 

Przeanalizujmy działanie funkcji analitycznych na przykładzie:

Znajdźmy kategorię (1. poziom), w której użytkownik dokonał 1. transakcji (jako kupujący)

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import rank, col

df = uam_orders.join(uam_offers, "offer_id").join(
    uam_categories, uam_categories.category_id == uam_offers.category_leaf
)

window = Window.partitionBy(df.buyer_id).orderBy(df.buyingTime)

display(
    df.withColumn("rank_", rank().over(window))
    .where(col("rank_") == 1)
    .select(df.buyer_id, df.category_level1)
)

A teraz trochę teorii dla uporządkowania:

Funkcje analityczne używamy analogicznie jak inne funkcje. Jedyną różnicą jest to, że po nazwie funkcji analitycznej wywołujemy metodę `over(window_spec)`, gdzie `window_spec` jest definicją okna.

`pyspark.sql.Window` posiada następujące metody:
1. `partitionBy(cols*)` - określenie kolumn, które wchodzą w skład definicji partycji
2. `orderBy(cols*)` - określenie sortowania wewnątrz partycji
3. `rangeBetween(start, end)` - definicja okna względem wartości danego wiersza
4. `rowsBetween(start, end)` - definicja okna względem pozycji danego wiersza

Stałe użyte w metodach `rangeBetween`, `rowsBetween` oznaczają: 
1. `Window.currentRow` -  bieżący wiersz
2. `Window.unboundedPreceding` - wszystkie wiersze poprzedzające bieżący wiersz
3. `Window.unboundedFollowing` - wszystkie wiersze następujące po bieżącym wierszu

Wybrane funkcje analityczne:

```
cume_dist()
dense_rank()
lag(col, count=1, default=None)
last(col, ignorenulls=False)
lead(col, count=1, default=None)
ntile(n)
percent_rank()
row_number()
rank()
```

Funkcje agregujące z definicją okna (np. `sum`, `max`, `count` itd.)

# Operacje na zbiorach 

Działania na zbiorach najlepiej zobrazują poniższe przykłady:

In [52]:
# Tworzymy następujące DF’y:
df1 = spark.range(0, 10, 2)  # liczby parzyste
df2 = spark.range(10)  # liczby naturalne

print("Suma zbiorów (może zawierać duplikaty)")
df1.union(df2).show()
print("Suma zbiorów (bez duplikatów)")
df1.union(df2).distinct().show()
print("Część wspólna zbiorów")
df2.intersect(df1).show()
print("Różnica zbiorów")
df2.subtract(df1).show()

Suma zbiorów (może zawierać duplikaty)
+---+
| id|
+---+
|  0|
|  2|
|  4|
|  6|
|  8|
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+

Suma zbiorów (bez duplikatów)
+---+
| id|
+---+
|  0|
|  2|
|  7|
|  6|
|  9|
|  5|
|  8|
|  4|
|  1|
|  3|
+---+

Część wspólna zbiorów
+---+
| id|
+---+
|  0|
|  2|
|  4|
|  6|
|  8|
+---+

Różnica zbiorów
+---+
| id|
+---+
|  1|
|  3|
|  5|
|  7|
|  9|
+---+



Ćwiczenie

Znajdź oferty, które nie znalazły nabywcy (identyfikatory) korzystając z operacji na zbiorach

In [58]:
display(
    uam_offers.select('offer_id')
    .subtract(
        uam_orders.select('offer_id')
    )
)

,offer_id
0,004172c9c5049a158006863afb8a3d11
1,0081e4d901216584e2fdc42a82ca02b3
2,008b51675b5cfc5601ccb2f2d029ef73
3,00cb7098fc5f8eea80d32032f60f26f3
4,00d100a15d9adaaabdca7ffac0f8d133
...,...
95,0af8da03e3ea44d53ea7cfe28d159d0c
96,0b35e59026d58f95f567bce09909caa4
97,0b3d96d550da89ce1980b3cb47b70291
98,0b4b1c94d85bd697db75851643e09135


Rozwiązanie

In [ ]:
display(
    uam_offers.select(uam_offers.offer_id).subtract(
        uam_orders.select(uam_orders.offer_id)
    )
)

# Operacja `cache`

Metoda `cache`/ `persist` służy do przechowywania danych w pamięci operacyjnej. Dzięki temu możemy uniknąć wykonywania tych samych akcji wielokrotnie. W PySparku możemy używać `cache` na poziomie dowolnego DataFrame'a w odróżnieniu do Spark SQL, gdzie działaliśmy na poziomie tabeli. Przejdźmy do przykładów.

Aby wyczyścić wszystkie dane z pamięci podręcznej możemy wykonać:

In [ ]:
spark.catalog.clearCache()

Utwórzmy DataFrame'a, który zawiera oferty będące ofertami z działu "Kolekcje i sztuka":

In [ ]:
df = uam_offers.join(
    uam_categories, uam_offers.category_leaf == uam_categories.category_id
).where(uam_categories.category_level1 == "Kolekcje i sztuka").select(
  "uam_offers.*"
)

In [ ]:
display(df)

Sprawdźmy, czy nasz DataFrame jest w cache'u:

In [ ]:
df.is_cached

Następnie po operacji cache:

In [ ]:
cached_df = df.cache()

In [ ]:
cached_df.is_cached

In [ ]:
display(cached_df)

Zauważmy, że optimalizator Sparka jest w stanie stwierdzić, które dane są w cache'u na poziomie analizy zapytania. W tym przypadku użyliśmy DataFrame'a, na którym nie wykonano operacji `cache`, ale mimo wszystko optymalizator znalazł odpowiednie dane w pamięci podręcznej i jest w stanie je wykorzystać:

In [ ]:
df.is_cached

Możemy usunąć dane z pamięci podręcznej

In [ ]:
uncached_df = cached_df.unpersist()
display(uncached_df)

In [ ]:
uncached_df.is_cached

In [ ]:
df.is_cached

In [ ]:
cached_df.is_cached

W związku z tym, że Spark korzysta z zaawansowanych mechanizmów takich jak:
  * CBO - Cost Based Optimizer (optymalizator kosztowy)
  * AQE - Adaptive Query Execution (adaptacyjne wykonanie zapytania)
  * Delta cache - pamięć podręczna dla plików w formacie Delta

plan zapytania może się różnić w zależności od wolumenów danych i parametryzacji Spark'a.

# DataFrame jako widok tymczasowy

A co jeśli odpowiada nam bardziej SQL API? Możemy zarejestrować DF jako widok tymczasowy:

In [ ]:
unsold_offers_df = uam_offers.select(uam_offers.offer_id).subtract(
    uam_orders.select(uam_orders.offer_id)
)
unsold_offers_df.createOrReplaceTempView("unsold_offers")

In [ ]:
%%sql
show tables

In [ ]:
%%sql
select
  count(1)
from
  unsold_offers

In [ ]:
display(spark.sql("select count(*) from unsold_offers"))

Może to być bardzo wygodne przy wykonywaniu zapytań z różnych źródeł (np. HDFS/DBFS i JDBC)

# Zapis danych

Do zapisu danych służy klasa `pyspark.sql.DataFrameWriter`, która jest tworzona poprzez wywołanie metody `write()` klasy `DataFrame`. Najważniejsze metody to:

`saveAsTable(name, format=None, mode=None, partitionBy=None, **options)`
1. `name` - nazwa tabeli
2. `mode` - tryb zapisu: 

 *append*: dodaje nowe wiersze
 
 *overwrite*: nadpisuje dane
 
 *ignore*: jeśli dane istnieją w tabeli nie robi nic
 
 *error* (domyślnie): zwraca wyjątek, jeśli dane istnieją
 
3. *partitionBy* - nazwy kolumn z partycjami
4. `**options` - inne opcje

`insertInto(tableName, overwrite=False)`
1. `tableName` - nazwa tabeli
2. `overwrite` - czy nadpisywać dane

Przeanalizujmy następujące przykłady

In [ ]:
# Usuwanie danych - potrzebne, gdy kilka razy wykonujemy polecenia w notebook'u
dbutils.fs.rm("/user/hive/warehouse/uam_categories_sample/", True)

In [ ]:
spark.sql("drop table if exists default.uam_categories_sample")

# zapis danych do nowej tabeli
uam_categories.sample(fraction=0.1).write.saveAsTable(
    "default.uam_categories_sample", format="orc"
)
# Dodanie nowych rekordów do tabeli
uam_categories.sample(fraction=0.1).write.insertInto("default.uam_categories_sample")

# Zapis danych do tabeli partycjonowanej
uam_categories.sample(fraction=0.1).write.saveAsTable(
    "default.uam_categories_sample",
    format="orc",
    mode="overwrite",
    partitionedBy=["category_level1"],
)

Dane możemy również zapisywać bezpośrednio do plików:

In [ ]:
uam_offers.sample(False, 0.01).write.partitionBy("duration").mode("overwrite").parquet(
    path="s3a://landing-zone/tmp/uam_offers_sample"
)

Analogicznie zapis może być dokonany do formatów: *orc, csv, text, json* oraz do źródeł danych *JDBC* (metoda `jdbc`)

W przypadku, gdy mamy zdefiniowaną tabelę w metastore, ale zdecydowaliśmy się pisać bezpośrednio do plików (jak w powyższym przykładzie), to dane mogę nie być widoczne. Poniższe polecenie sprawi, że metastore odświeży swoje metadane i uwzględni nowe pliki:
```python
spark.catalog.recoverPartitions(nazwa_tabeli)
```
Powyższe polecenie jest odpowiednikiem SQL'owego `MSCK REPAIR TABLE ...`
<br><br><br>
Czasami może się zdarzyć, że odczytujemy dane z pewnej tabeli, potem ją modyfikujemy (albo robi inny program), a potem znowu próbujemy ją odczytać. W przypadku, gdy spark zwróci wyjątek należy wykonać:
```python
spark.catalog.refreshTable(nazwa_tabeli)
```
Jest to odpowiednik SQL'owego `REFRESH TABLE ...`

# Pandas & Koalas

Biblioteka pandas jest jedną z najpolularniejszych bibliotek pythonowych do analizy danych (zob.: https://pandas.pydata.org/). Aby z niech korzystać wystarczy wywołać metodą `toPandas`:

In [ ]:
uam_categories.limit(10).toPandas()

Warto pamiętać że `toPandas` jest akcją. Oznacza to, że wszystkie dane są zbierane do pamięcie drivera (działa podobnie jak `collect`). Od najnowszych wersji Spark'a (3.2) możemy korzystać z tzw. Pandas API. Zapewnia ono podobne API jak `pandas`, ale przetwarzania mają charakter rozproszony. Oznacza to, że nie ogranicza nas pamięć driver'a :) Poniżej znajduje się przykład, w którym jest tworzony tzw. `pandas-on-Spark DataFrame`.

Przykład:

In [ ]:
pandas_on_spark_df = uam_categories.pandas_api()

W łatwy sposób możemy wrócić do Spark'owego DataFrame'a:

In [ ]:
df = pandas_on_spark_df.to_spark()

Poniżej przykład, który pokazuje możliwości Pandas API:

In [ ]:
pandas_on_spark_df[["category_id","category_level1"]].iloc[:4]

Dane są wyświetlane na ekran. Oznacza, że mamy do czynienia z akcją.

# Testowanie

[Tutaj](https://spark.apache.org/docs/latest/api/python/getting_started/testing_pyspark.html) można znaleźć więcej przykładów osadzonych w kontekście popularnych frameworków do tetowania tj. `pytest` i `unittest`.

Skupmy się na prostym przykładzie jak można sprawdzić, że nasz kod sparkowy przekształca dane zgodnie z oczekiwaniem.

Przykład

Oryginalne zapytanie stworzone w notebooku nie jest dobrym przykładem kodu, który łatwo testować:

In [59]:
from pyspark.sql.functions import min, max

# Przykład grupowania z części dot groupBy:
# Min, Max cena oferty per kategoria
(
    uam_offers.join(
        uam_categories, uam_categories.category_id == uam_offers.category_leaf
    )
    .groupBy(uam_categories.category_level1)
    .agg(
        min(uam_offers.buynow_price.cast("double")).alias("min_price"),
        max(uam_offers.buynow_price.cast("double")).alias("max_price"),
    )
)


DataFrame[category_level1: string, min_price: double, max_price: double]

Zmieńmy je zgodnie z dobrymi praktykami tworząc fukcję. 

UWAGA: tego typu kod raczej nie tworzymy w notebookach, ale korzystając z IDE (PyCharm, VS Code).

In [60]:
def offer_stats_per_category(offers, categories):
  return (
      offers.join(
      categories, categories.category_id == offers.category_leaf
    )
    .groupBy(categories.category_level1)
    .agg(
        min(offers.buynow_price.cast("double")).alias("min_price"),
        max(offers.buynow_price.cast("double")).alias("max_price"),
    )
  )

Sprawdźmy działanie:

In [61]:
display(
  offer_stats_per_category(uam_offers, uam_categories)
)

,category_level1,min_price,max_price
0,Kultura i rozrywka,1.00,218.90
1,Firma i usługi,1.49,3688.77
2,Dziecko,5.99,319.00
3,Dom i zdrowie,1.11,799.00
4,Kolekcje i sztuka,4.75,199.90
5,Motoryzacja,4.00,399.99
6,Sport i wypoczynek,3.80,147.99
7,Moda i uroda,1.40,773.00
8,Elektronika,3.90,2679.00


Przejdźmy do testowania. W takim przypadku nie bazujemu na danych w tabelach. Musimy przygotować przypadki dane testowe:

In [62]:
offer_test_df = spark.createDataFrame(
  [
    # Możemy ograniczyć się do kolumn występujących w DF, których używamy w przetwarzaniu
    {"category_leaf": 10, "buynow_price": 10.00},
    {"category_leaf": 20, "buynow_price": 20.00},
    {"category_leaf": 20, "buynow_price": 30.00},
    {"category_leaf": 20, "buynow_price": 40.00},
    {"category_leaf": 10, "buynow_price": 100.00},
  ]
)

categories_test_df = spark.createDataFrame(
  [
    # Możemy ograniczyć się do kolumn występujących w DF, których używamy w przetwarzaniu
    {"category_id": 10, "category_level1": "kat A"},
    {"category_id": 20, "category_level1": "kat B"},
    {"category_id": 30, "category_level1": "kat C"},
  ]
)

Zobaczmy jak to działa:

In [63]:
display(
  offer_stats_per_category(offer_test_df, categories_test_df)
)

,category_level1,min_price,max_price
0,kat B,20.0,40.0
1,kat A,10.0,100.0


In [64]:
offer_stats_per_category(offer_test_df, categories_test_df)

DataFrame[category_level1: string, min_price: double, max_price: double]

Nie chcemy jednak testować kodu oczami! Wykorzystajmy funkcję `assertDataFrameEqual`:

In [65]:
from pyspark.testing.utils import assertDataFrameEqual

expected_output_df = spark.createDataFrame(
  [
    {"category_level1": "kat A", "max_price": 100.0, "min_price": 10.0},
    {"category_level1": "kat B", "max_price": 40.0, "min_price": 20.0},
  ],
)

result_df = offer_stats_per_category(offer_test_df, categories_test_df)

# kolejność kolumn ma znaczenie, przy porównywaniu DF
columns_to_check = ["category_level1", "max_price", "min_price"]

assertDataFrameEqual(
  actual=result_df.select(*columns_to_check),
  expected=expected_output_df.select(*columns_to_check)
)


/opt/spark/python/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


Ćwiczenie

Zmodyfikuj `offer_stats_per_category`, aby dodatkowo wyliczana była średnia cena. 
Zmodyfikuj również testy, aby uwzględniały nowe wymaganie.

In [67]:
from pyspark.sql.functions import min, max, avg


def offer_stats_per_category(offers, categories):
  return (
      offers.join(
      categories, categories.category_id == offers.category_leaf
    )
    .groupBy(categories.category_level1)
    .agg(
        min(offers.buynow_price.cast("double")).alias("min_price"),
        max(offers.buynow_price.cast("double")).alias("max_price"),
        avg(offers.buynow_price.cast("double")).alias("avg_price"),
    )
  )

Rozwiązanie

In [76]:
# Kod produkcyjny

import pyspark.sql.functions as F 

def offer_stats_per_category(offers, categories):
  return (
      offers.join(
      categories, categories.category_id == offers.category_leaf
    )
    .groupBy(categories.category_level1)
    .agg(
        F.min(offers.buynow_price.cast("double")).alias("min_price"),
        F.max(offers.buynow_price.cast("double")).alias("max_price"),
        F.avg(offers.buynow_price.cast("double")).alias("avg_price"),
    )
  )

In [77]:
# Kod testowy

# GIVEN:
# bez zmian :)
offer_test_df = spark.createDataFrame(
  [
    {"category_leaf": 10, "buynow_price": 10.00},
    {"category_leaf": 20, "buynow_price": 20.00},
    {"category_leaf": 20, "buynow_price": 30.00},
    {"category_leaf": 20, "buynow_price": 40.00},
    {"category_leaf": 10, "buynow_price": 100.00},
  ]
)

categories_test_df = spark.createDataFrame(
  [
    {"category_id": 10, "category_level1": "kat A"},
    {"category_id": 20, "category_level1": "kat B"},
    {"category_id": 30, "category_level1": "kat C"},
  ]
)

# WHEN:
result_df = offer_stats_per_category(offer_test_df, categories_test_df)

# THEN:
expected_output_df = spark.createDataFrame(
  [
    {"category_level1": "kat A", "max_price": 100.0, "min_price": 10.0, "avg_price": 55.0},
    {"category_level1": "kat B", "max_price": 40.0, "min_price": 20.0, "avg_price": 30.0},
  ],
)

columns_to_check = ["category_level1", "max_price", "min_price"]

assertDataFrameEqual(
  actual=result_df.select(*columns_to_check),
  expected=expected_output_df.select(*columns_to_check)
)

# Porównanie różnych API

In [69]:
# 1) RDD - imperatywne API
uam_categories.rdd.filter(lambda r: r.category_level1 == "Elektronika").map(
    lambda r: (r.category_level2, 1)
).reduceByKey(lambda v1, v2: v1 + v2).take(10)

[('Komputery', 6),
 ('Telefony i Akcesoria', 16),
 ('RTV i AGD', 6),
 ('Fotografia', 2)]

In [70]:
# 2) Spark SQL - deklaratywne API
display(
    spark.sql(
        """
          select category_level2,  count(1)
          from uam_categories where category_level1 = 'Elektronika'
          group by category_level2
        """.strip()
    )
)

,category_level2,count(1)
0,RTV i AGD,6
1,Fotografia,2
2,Telefony i Akcesoria,16
3,Komputery,6


In [71]:
# 3) DataFrame - deklaratywne API
display(
    uam_categories.where(uam_categories.category_level1 == "Elektronika")
    .groupBy("category_level2")
    .count()
)

,category_level2,count
0,RTV i AGD,6
1,Fotografia,2
2,Telefony i Akcesoria,16
3,Komputery,6


In [72]:
# 4) Pandas API
pandas_uam_categories = uam_categories.pandas_api()

pandas_uam_categories[pandas_uam_categories.category_level1 == "Elektronika"].groupby(
    ["category_level2"]
).size()

category_level2
RTV i AGD                6
Fotografia               2
Telefony i Akcesoria    16
Komputery                6
dtype: int64

In [78]:
spark.stop()